# Chapter 3: Error Analysis
Steps: make test queries → run the system → read traces (open coding) → group notes (axial coding) → label and count (prevalence).

## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {'ch3': {'dimensions': {'feature': ['property search', 'scheduling', 'email drafting'], 'client_persona': ['first-time buyer', 'investor', 'luxury buyer'], 'scenario': ['clear request', 'vague request', 'out-of-scope request']}, 'query_prompt': 'You write one realistic message that a user would send to a real estate assistant.\nFeature: {feature}\nClient persona: {client_persona}\nScenario: {scenario}\nReturn only the message text.', 'traces': [{'id': 1, 'query': 'Find pet-friendly 2-bed apartments under $3k in Austin', 'response': 'Here is a 2-bed for $2,800 near downtown (no pets policy listed).', 'open_note': 'Dropped the pet-friendly filter'}, {'id': 2, 'query': 'Schedule a showing for Saturday 10am at 12 Oak St', 'response': 'Booked for Sunday 10am at 12 Oak St.', 'open_note': 'Booked the wrong day'}, {'id': 3, 'query': 'Draft an email to a seller on behalf of my luxury client', 'response': 'Hey! Awesome house, my client is super keen, lol.', 'open_note': 'Slang tone for a luxury buyer'}, {'id': 4, 'query': 'Show me good properties near the beach', 'response': 'Here are 5 properties near the beach.', 'open_note': "Did not ask what 'good' means"}, {'id': 5, 'query': 'What is the weather in Paris this weekend?', 'response': 'It will be sunny and 22C in Paris.', 'open_note': 'Answered an out-of-scope question'}, {'id': 6, 'query': 'Show 3-bed houses under 400k in Dallas', 'response': 'Here are two 3-bed houses at 380k and 395k in Dallas.', 'open_note': ''}, {'id': 7, 'query': 'Houses under 400k with 3 beds', 'response': 'Found a 4-bed house for 390k.', 'open_note': 'Ignored the bedroom count'}, {'id': 8, 'query': 'Book a viewing tomorrow at 2pm for 5 Pine Rd', 'response': 'Booked for 2pm today at 5 Pine Rd.', 'open_note': 'Booked the wrong day'}, {'id': 9, 'query': 'List duplexes in Phoenix with yearly rent above 30k', 'response': 'Here are 3 duplexes in Phoenix with yearly rent of 32k, 35k and 41k.', 'open_note': ''}, {'id': 10, 'query': 'Write an email to my investor client about a duplex', 'response': 'Hey dude!! Hot deal, you will love it!!!', 'open_note': 'Slang tone'}], 'axial_map': {'Dropped the pet-friendly filter': 'missing_constraint', 'Ignored the bedroom count': 'missing_constraint', 'Booked the wrong day': 'wrong_date', 'Slang tone for a luxury buyer': 'wrong_tone', 'Slang tone': 'wrong_tone', "Did not ask what 'good' means": 'no_clarification', 'Answered an out-of-scope question': 'out_of_scope_not_refused'}}}


In [ ]:
import itertools, random
import pandas as pd

data = load_data()["ch3"]

## 1. Synthetic queries: dimensions → tuples → queries
**Dimensions** = ways requests can differ. A **tuple** = one value from each dimension.

In [ ]:
dims = data["dimensions"]
for name, values in dims.items():
    print(name, "->", values)

all_tuples = list(itertools.product(*dims.values()))
print("\nTotal possible tuples:", len(all_tuples))

random.seed(1)
chosen = random.sample(all_tuples, 5)           # pick 5 to test
for t in chosen:
    print(t)

Now ask the LLM to write one realistic user message per tuple.

In [ ]:
queries = []
for feature, persona, scenario in chosen:
    prompt = (data["query_prompt"]
              .replace("{feature}", feature)
              .replace("{client_persona}", persona)
              .replace("{scenario}", scenario))
    q = ask(prompt, temperature=0.7)
    queries.append(q)
    print(f"({feature}, {persona}, {scenario})\n  -> {q}\n")

## 2. Look at traces
A **trace** = the full record of one run (query + response; real apps also have tool calls). We use 10 ready-made traces from the JSON file.

In [ ]:
traces = data["traces"]
for t in traces:
    print(f"#{t['id']} Q: {t['query']}\n    A: {t['response']}\n")

## 3. Open coding
Read each trace and write a short note about the **first thing that went wrong**. Empty note = no problem.

In [ ]:
for t in traces:
    print(f"#{t['id']:<3} note: {t['open_note'] or '(no failure)'}")

## 4. Axial coding
Group similar notes under one **failure mode** name (this list is your *taxonomy*).

In [ ]:
axial_map = data["axial_map"]       # note -> failure mode
for note, mode in axial_map.items():
    print(f"{note:40} -> {mode}")

taxonomy = sorted(set(axial_map.values()))
print("\nTaxonomy:", taxonomy)

## 5. Labeling table + prevalence
One column per failure mode: `1` = happened, `0` = did not. **Prevalence** = traces with failure ÷ all traces.

In [ ]:
rows = []
for t in traces:
    mode = axial_map.get(t["open_note"])             # None if no failure
    row = {"trace": t["id"]}
    for m in taxonomy:
        row[m] = 1 if mode == m else 0
    rows.append(row)

labels = pd.DataFrame(rows).set_index("trace")
display(labels)

prevalence = labels.mean().sort_values(ascending=False)
print("Prevalence (share of traces):")
print((prevalence * 100).round(0).astype(int).astype(str) + "%")

## 6. Saturation check
Walk through the traces one by one. When do **new** failure modes stop appearing?

In [ ]:
seen = set()
for t in traces:
    mode = axial_map.get(t["open_note"])
    if mode and mode not in seen:
        seen.add(mode)
        print(f"trace #{t['id']}: NEW failure mode -> {mode}")
print("\nTotal modes found:", len(seen), "| If the last 20 traces show nothing new, you are done (book suggests ~100 traces).")